# Advanced Problems with Solutions: List Comprehensions

This notebook gives advanced, runnable practice for Python list comprehensions.

Topics covered:

- transforming iterables into lists
- filtering with `if`
- conditional expressions inside comprehensions
- nested comprehensions
- nested loop comprehensions
- `zip`, `enumerate`, and `range`
- flattening data
- matrix-style data
- scope behavior
- closures and lambda capture pitfalls
- bytecode inspection with `compile()` and `dis`
- readability and best practices


## Best-practice reminders

A good list comprehension is usually:

- short enough to read at a glance
- focused on building one list
- clear about transformation and filtering
- not overloaded with too much business logic

A comprehension is often a great replacement for:

```python
result = []
for item in iterable:
    if condition(item):
        result.append(transform(item))
```

But if the logic becomes hard to read, use a normal loop or helper function.


## Problem 1 — Basic transformation plus filtering

Build a list containing the squares of numbers from `1` to `50`, but only for numbers divisible by `3` or `5`.

Then solve the same task using a traditional loop so you can compare both approaches.


In [1]:
# Traditional loop solution
squares_loop = []
for n in range(1, 51):
    if n % 3 == 0 or n % 5 == 0:
        squares_loop.append(n ** 2)
# List comprehension solution
squares_comp = [n ** 2 for n in range(1, 51) if n % 3 == 0 or n % 5 == 0]
expected = [
    9, 25, 36, 81, 100, 144, 225, 324, 400, 441,
    576, 625, 729, 900, 1089, 1225, 1296, 1521, 1600,
    1764, 2025, 2304, 2500,
]
assert squares_loop == expected
assert squares_comp == expected
squares_comp


[9,
 25,
 36,
 81,
 100,
 144,
 225,
 324,
 400,
 441,
 576,
 625,
 729,
 900,
 1089,
 1225,
 1296,
 1521,
 1600,
 1764,
 2025,
 2304,
 2500]

### Solution notes

The expression before `for` is the transformation: `n ** 2`.
The `if` at the end is the filter: only numbers divisible by `3` or `5` are kept.


## Problem 2 — Filtering vs conditional expressions

Given a list of integers, produce two outputs:

1. `positive_evens`: keep only positive even numbers.
2. `labels`: keep all numbers, but replace each number with `'positive'`, `'negative'`, or `'zero'`.

This problem highlights the difference between a filtering `if` and an inline conditional expression.


In [2]:
numbers = [-5, -2, 0, 1, 4, 7, 10, -8, 12]

positive_evens = [n for n in numbers if n > 0 and n % 2 == 0]

labels = [
    "positive" if n > 0 else "negative" if n < 0 else "zero"
    for n in numbers
]

assert positive_evens == [4, 10, 12]
assert labels == [
    "negative", "negative", "zero", "positive", "positive",
    "positive", "positive", "negative", "positive",
]

positive_evens, labels


([4, 10, 12],
 ['negative',
  'negative',
  'zero',
  'positive',
  'positive',
  'positive',
  'positive',
  'negative',
  'positive'])

### Solution notes

A filtering `if` appears after the `for` part and decides whether the item is included.
A conditional expression appears before the `for` part and decides what value to put into the output list.


## Problem 3 — Data cleaning with comprehensions

You receive messy user names. Build a clean list where each name is:

- stripped of leading/trailing spaces
- converted to title case
- included only if it is not empty after stripping

Then create a second list containing only names with at least five characters.


In [3]:
raw_names = [" alice ", "", "BOB", "   carol", "dave   ", "   ", "eve", "FRANK"]

clean_names = [name.strip().title() for name in raw_names if name.strip()]
long_names = [name for name in clean_names if len(name) >= 5]

assert clean_names == ["Alice", "Bob", "Carol", "Dave", "Eve", "Frank"]
assert long_names == ["Alice", "Carol", "Frank"]

clean_names, long_names


(['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank'],
 ['Alice', 'Carol', 'Frank'])

### Solution notes

This is a useful pattern, but notice that `name.strip()` is repeated. For simple cases that is fine, but for heavier work a helper function or loop may be clearer.


## Problem 4 — Avoid repeated work with a helper function

Improve the previous cleaning example by using a helper function.

Return names that are valid after cleaning. A valid cleaned name must:

- not be empty
- contain only alphabetic characters
- have length at least `3`

Use a list comprehension, but keep the validation readable.


In [4]:
def clean_name(value):
    return value.strip().title()


def is_valid_name(value):
    return value.isalpha() and len(value) >= 3


raw_names = [" alice ", "b2b", "BOB", "   ", "ed", "frank", "maria-jose", "NORA"]

cleaned_valid_names = [
    cleaned
    for cleaned in [clean_name(name) for name in raw_names]
    if is_valid_name(cleaned)
]

assert cleaned_valid_names == ["Alice", "Bob", "Frank", "Nora"]
cleaned_valid_names


['Alice', 'Bob', 'Frank', 'Nora']

### Solution notes

The inner comprehension performs the transformation. The outer comprehension filters the transformed values.

For even better readability in production code, a normal loop may be preferable when validation becomes complex.


## Problem 5 — Nested loop comprehension: Cartesian products

Given shirt colors and sizes, generate all product SKUs in the format `COLOR-SIZE`.

Then exclude unavailable combinations.


In [5]:
colors = ["black", "white", "blue"]
sizes = ["S", "M", "L", "XL"]
unavailable = {("white", "XL"), ("blue", "S")}

all_skus = [f"{color.upper()}-{size}" for color in colors for size in sizes]

available_skus = [
    f"{color.upper()}-{size}"
    for color in colors
    for size in sizes
    if (color, size) not in unavailable
]

assert len(all_skus) == 12
assert "WHITE-XL" in all_skus
assert "WHITE-XL" not in available_skus
assert "BLUE-S" not in available_skus
assert available_skus == [
    "BLACK-S", "BLACK-M", "BLACK-L", "BLACK-XL",
    "WHITE-S", "WHITE-M", "WHITE-L",
    "BLUE-M", "BLUE-L", "BLUE-XL",
]

available_skus


['BLACK-S',
 'BLACK-M',
 'BLACK-L',
 'BLACK-XL',
 'WHITE-S',
 'WHITE-M',
 'WHITE-L',
 'BLUE-M',
 'BLUE-L',
 'BLUE-XL']

### Solution notes

The order of `for` clauses in a comprehension matches the order of nested loops:

```python
for color in colors:
    for size in sizes:
        ...
```


## Problem 6 — Nested comprehension: multiplication table

Create a `12 x 12` multiplication table using a nested list comprehension.

Then extract:

- the fifth row
- the diagonal values
- all values divisible by both `3` and `4`


In [6]:
table = [[row * col for col in range(1, 13)] for row in range(1, 13)]

fifth_row = table[4]
diagonal = [table[i][i] for i in range(len(table))]
divisible_by_12 = [value for row in table for value in row if value % 12 == 0]

assert fifth_row == [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60]
assert diagonal == [1, 4, 9, 16, 25, 36, 49, 64, 81, 100, 121, 144]
assert divisible_by_12[:10] == [12, 12, 24, 12, 24, 36, 12, 24, 36, 48]

fifth_row, diagonal, divisible_by_12[:15]



([5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60],
 [1, 4, 9, 16, 25, 36, 49, 64, 81, 100, 121, 144],
 [12, 12, 24, 12, 24, 36, 12, 24, 36, 48, 60, 12, 24, 36, 48])

### Solution notes

`[[... for col ...] for row ...]` creates a list of rows.
`[value for row in table for value in row]` flattens a list of rows.


## Problem 7 — Matrix transpose

Transpose a matrix using a list comprehension.

Example:

```python
[[1, 2, 3],
 [4, 5, 6]]
```

becomes:

```python
[[1, 4],
 [2, 5],
 [3, 6]]
```


In [7]:
matrix = [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
]

transpose = [
    [row[col_index] for row in matrix]
    for col_index in range(len(matrix[0]))
]

expected = [
    [1, 5, 9],
    [2, 6, 10],
    [3, 7, 11],
    [4, 8, 12],
]

assert transpose == expected
transpose


[[1, 5, 9], [2, 6, 10], [3, 7, 11], [4, 8, 12]]

### Solution notes

The outer comprehension chooses each column index. The inner comprehension collects that column's value from every row.


## Problem 8 — Flattening nested records

You have course grade data stored as a dictionary of course names to student-score pairs.

Build a flat list of dictionaries with the keys:

- `course`
- `student`
- `score`
- `passed`

A score of `70` or above is passing.


In [8]:
grades = {
    "Python": [("Alice", 92), ("Bob", 64), ("Carol", 81)],
    "Math": [("Alice", 75), ("Bob", 88), ("Carol", 59)],
    "Databases": [("Alice", 68), ("Bob", 72), ("Carol", 90)],
}

flat_records = [
    {
        "course": course,
        "student": student,
        "score": score,
        "passed": score >= 70,
    }
    for course, rows in grades.items()
    for student, score in rows
]

passed_records = [record for record in flat_records if record["passed"]]
failed_records = [record for record in flat_records if not record["passed"]]

assert len(flat_records) == 9
assert len(passed_records) == 6
assert len(failed_records) == 3
assert failed_records == [
    {"course": "Python", "student": "Bob", "score": 64, "passed": False},
    {"course": "Math", "student": "Carol", "score": 59, "passed": False},
    {"course": "Databases", "student": "Alice", "score": 68, "passed": False},
]

flat_records


[{'course': 'Python', 'student': 'Alice', 'score': 92, 'passed': True},
 {'course': 'Python', 'student': 'Bob', 'score': 64, 'passed': False},
 {'course': 'Python', 'student': 'Carol', 'score': 81, 'passed': True},
 {'course': 'Math', 'student': 'Alice', 'score': 75, 'passed': True},
 {'course': 'Math', 'student': 'Bob', 'score': 88, 'passed': True},
 {'course': 'Math', 'student': 'Carol', 'score': 59, 'passed': False},
 {'course': 'Databases', 'student': 'Alice', 'score': 68, 'passed': False},
 {'course': 'Databases', 'student': 'Bob', 'score': 72, 'passed': True},
 {'course': 'Databases', 'student': 'Carol', 'score': 90, 'passed': True}]

### Solution notes

This is a nested loop comprehension over dictionary items and then over each course's student-score rows.


## Problem 9 — Recreate part of `zip` with a comprehension

Implement `zip_lists(left, right)` using a list comprehension.

Requirements:

- pair items by index
- stop at the shorter input
- return a list of tuples

Do not use the built-in `zip` inside your function.


In [9]:
def zip_lists(left, right):
    limit = min(len(left), len(right))
    return [(left[i], right[i]) for i in range(limit)]


l1 = [1, 2, 3, 4, 5]
l2 = ["a", "b", "c"]

result = zip_lists(l1, l2)

assert result == [(1, "a"), (2, "b"), (3, "c")]
assert result == list(zip(l1, l2))
result


[(1, 'a'), (2, 'b'), (3, 'c')]

### Solution notes

The built-in `zip` is simpler and more general, but this exercise helps connect indexing, `range`, and comprehensions.


## Problem 10 — Dot product and vector operations

Given two equal-length vectors, compute:

1. the dot product
2. the element-wise sums
3. the squared differences
4. the squared Euclidean distance

Use comprehensions with `zip`.


In [10]:
v1 = (3, -2, 7, 4)
v2 = (1, 5, -3, 4)

products = [a * b for a, b in zip(v1, v2)]
dot_product = sum(products)

pairwise_sums = [a + b for a, b in zip(v1, v2)]
squared_diffs = [(a - b) ** 2 for a, b in zip(v1, v2)]
squared_distance = sum(squared_diffs)

assert products == [3, -10, -21, 16]
assert dot_product == -12
assert pairwise_sums == [4, 3, 4, 8]
assert squared_diffs == [4, 49, 100, 0]
assert squared_distance == 153

products, dot_product, pairwise_sums, squared_diffs, squared_distance


([3, -10, -21, 16], -12, [4, 3, 4, 8], [4, 49, 100, 0], 153)

### Solution notes

When using `zip`, the comprehension receives paired values directly, avoiding manual indexing.


## Problem 11 — Pascal's triangle

Build Pascal's triangle with `size = 8`.

Use:

```python
math.comb(n, k)
```

Then verify that the sum of row `n` is `2 ** n`.


In [11]:
from math import comb

size = 8
pascal = [[comb(n, k) for k in range(n + 1)] for n in range(size + 1)]
row_sums = [sum(row) for row in pascal]
expected_sums = [2 ** n for n in range(size + 1)]

assert pascal[0] == [1]
assert pascal[4] == [1, 4, 6, 4, 1]
assert pascal[8] == [1, 8, 28, 56, 70, 56, 28, 8, 1]
assert row_sums == expected_sums

pascal, row_sums


([[1],
  [1, 1],
  [1, 2, 1],
  [1, 3, 3, 1],
  [1, 4, 6, 4, 1],
  [1, 5, 10, 10, 5, 1],
  [1, 6, 15, 20, 15, 6, 1],
  [1, 7, 21, 35, 35, 21, 7, 1],
  [1, 8, 28, 56, 70, 56, 28, 8, 1]],
 [1, 2, 4, 8, 16, 32, 64, 128, 256])

### Solution notes

This is a nested comprehension where the inner range depends on the outer variable `n`.
The inner comprehension can access `n` from the enclosing comprehension scope.


## Problem 12 — Scope: comprehension variables do not leak

Show that a comprehension loop variable does not overwrite a variable with the same name in the surrounding scope.

Then show that a comprehension can still read variables from the surrounding scope.


In [12]:
number = 100
values = [number ** 2 for number in range(5)]

assert values == [0, 1, 4, 9, 16]
assert number == 100  # The outer variable was not overwritten.

factor = 10
scaled = [factor * n for n in range(5)]

assert scaled == [0, 10, 20, 30, 40]
assert factor == 10

values, number, scaled, factor


([0, 1, 4, 9, 16], 100, [0, 10, 20, 30, 40], 10)

### Solution notes

The loop variable in a list comprehension is local to the comprehension. But variables that are only read, such as `factor`, can come from the enclosing scope.


## Problem 13 — Closure pitfall with lambdas inside comprehensions

Create six functions that should compute:

- `x ** 0`
- `x ** 1`
- `x ** 2`
- ...
- `x ** 5`

First demonstrate the common bug. Then fix it using a default parameter.


In [13]:
# Buggy version: every lambda closes over the same variable i.
buggy_funcs = [lambda x: x ** i for i in range(6)]
buggy_results = [fn(10) for fn in buggy_funcs]

# Fixed version: current value of i is captured as default argument pow=i.
fixed_funcs = [lambda x, pow=i: x ** pow for i in range(6)]
fixed_results = [fn(10) for fn in fixed_funcs]

assert buggy_results == [100000, 100000, 100000, 100000, 100000, 100000]
assert fixed_results == [1, 10, 100, 1000, 10000, 100000]

buggy_results, fixed_results


([100000, 100000, 100000, 100000, 100000, 100000],
 [1, 10, 100, 1000, 10000, 100000])

### Solution notes

The buggy lambdas all reference the same `i`, whose final value is `5` after the comprehension finishes.
The fixed version stores each current value of `i` in a default parameter at function creation time.


## Problem 14 — Use `dis` to inspect a comprehension

Compile and disassemble this expression:

```python
[n ** 2 for n in range(3)]
```

Then check whether your Python version stores a separate `<listcomp>` code object.

Older Python versions commonly show a nested `<listcomp>` code object. Newer Python versions may inline the comprehension, so this notebook handles both cases.



In [14]:
import dis

expression = "[n ** 2 for n in range(3)]"
compiled = compile(expression, filename="<listcomp-demo>", mode="eval")

print("Outer compiled expression:")
dis.dis(compiled)

listcomp_objects = [
    const
    for const in compiled.co_consts
    if getattr(const, "co_name", None) == "<listcomp>"
]

if listcomp_objects:
    print("\nInner <listcomp> code object:")
    dis.dis(listcomp_objects[0])
else:
    print("\nNo separate <listcomp> code object found in this Python version.")

result = eval(compiled)
assert result == [0, 1, 4]
result



Outer compiled expression:
   0           RESUME                   0

   1           LOAD_NAME                0 (range)
               PUSH_NULL
               LOAD_CONST               0 (3)
               CALL                     1
               GET_ITER
               LOAD_FAST_AND_CLEAR      0 (n)
               SWAP                     2
       L1:     BUILD_LIST               0
               SWAP                     2
               GET_ITER
       L2:     FOR_ITER                 7 (to L3)
               STORE_FAST_LOAD_FAST     0 (n, n)
               LOAD_CONST               1 (2)
               BINARY_OP                8 (**)
               LIST_APPEND              2
               JUMP_BACKWARD            9 (to L2)
       L3:     END_FOR
               POP_TOP
       L4:     SWAP                     2
               STORE_FAST               0 (n)
               RETURN_VALUE

  --   L5:     SWAP                     2
               POP_TOP

   1           SWAP               

[0, 1, 4]

### Solution notes

Python implements a list comprehension using an internal function-like code object. This is why comprehension variables have their own local scope.


## Problem 15 — Readability refactor

The following comprehension works, but is hard to read:

```python
[(name.strip().title(), int(score)) for name, score in rows if name.strip() and score.isdigit() and int(score) >= 70]
```

Refactor it using helper functions while still using a final list comprehension.


In [15]:
rows = [
    (" alice ", "92"),
    ("", "85"),
    ("bob", "not-a-score"),
    ("carol", "69"),
    (" dave", "70"),
    ("EVE ", "100"),
]


def normalize_row(row):
    name, score_text = row
    return name.strip().title(), score_text.strip()


def is_passing_row(row):
    name, score_text = row
    return bool(name) and score_text.isdigit() and int(score_text) >= 70


def convert_row(row):
    name, score_text = row
    return {"name": name, "score": int(score_text)}


normalized_rows = [normalize_row(row) for row in rows]
passing_students = [
    convert_row(row)
    for row in normalized_rows
    if is_passing_row(row)
]

assert passing_students == [
    {"name": "Alice", "score": 92},
    {"name": "Dave", "score": 70},
    {"name": "Eve", "score": 100},
]

passing_students


[{'name': 'Alice', 'score': 92},
 {'name': 'Dave', 'score': 70},
 {'name': 'Eve', 'score': 100}]

### Solution notes

Comprehensions are not only about making code shorter. They should make code clearer. Helper functions often make advanced comprehensions easier to understand and test.


## Problem 16 — Comprehensions with dictionaries and grouping

Given a list of orders, build:

1. a list of order totals
2. a list of completed order IDs
3. a list of expensive completed orders, where total is at least `100`
4. a list of customer names for completed orders, normalized to title case


In [16]:
orders = [
    {"id": "o100", "customer": "alice", "items": [35, 20, 10], "status": "complete"},
    {"id": "o101", "customer": "bob", "items": [200], "status": "pending"},
    {"id": "o102", "customer": "carol", "items": [50, 55], "status": "complete"},
    {"id": "o103", "customer": "dave", "items": [12, 8, 5], "status": "cancelled"},
    {"id": "o104", "customer": "eve", "items": [80, 25], "status": "complete"},
]

order_totals = [(order["id"], sum(order["items"])) for order in orders]
completed_ids = [order["id"] for order in orders if order["status"] == "complete"]
expensive_completed = [
    order["id"]
    for order in orders
    if order["status"] == "complete" and sum(order["items"]) >= 100
]
completed_customers = [
    order["customer"].title()
    for order in orders
    if order["status"] == "complete"
]

assert order_totals == [("o100", 65), ("o101", 200), ("o102", 105), ("o103", 25), ("o104", 105)]
assert completed_ids == ["o100", "o102", "o104"]
assert expensive_completed == ["o102", "o104"]
assert completed_customers == ["Alice", "Carol", "Eve"]

order_totals, completed_ids, expensive_completed, completed_customers


([('o100', 65), ('o101', 200), ('o102', 105), ('o103', 25), ('o104', 105)],
 ['o100', 'o102', 'o104'],
 ['o102', 'o104'],
 ['Alice', 'Carol', 'Eve'])

### Solution notes

List comprehensions work well with lists of dictionaries, but repeated expressions like `sum(order["items"])` may be worth assigning in a loop or helper function if the logic grows.


## Problem 17 — Build a board with coordinates

Create all squares on a chessboard using a list comprehension.

Then build:

1. all white squares
2. all black squares
3. all legal rook moves from `d4`

Use coordinates like `'a1'`, `'h8'`.


In [17]:
files = "abcdefgh"
ranks = range(1, 9)

squares = [f"{file}{rank}" for rank in ranks for file in files]

white_squares = [
    f"{file}{rank}"
    for rank in ranks
    for file_index, file in enumerate(files, start=1)
    if (file_index + rank) % 2 == 0
]

black_squares = [square for square in squares if square not in white_squares]

origin = "d4"
origin_file = origin[0]
origin_rank = int(origin[1])
rook_moves = [
    square
    for square in squares
    if square != origin and (square[0] == origin_file or int(square[1]) == origin_rank)
]

assert len(squares) == 64
assert len(white_squares) == 32
assert len(black_squares) == 32
assert len(rook_moves) == 14
assert "d1" in rook_moves
assert "a4" in rook_moves
assert "e5" not in rook_moves

squares[:8], white_squares[:8], rook_moves


(['a1', 'b1', 'c1', 'd1', 'e1', 'f1', 'g1', 'h1'],
 ['a1', 'c1', 'e1', 'g1', 'b2', 'd2', 'f2', 'h2'],
 ['d1',
  'd2',
  'd3',
  'a4',
  'b4',
  'c4',
  'e4',
  'f4',
  'g4',
  'h4',
  'd5',
  'd6',
  'd7',
  'd8'])

### Solution notes

This problem combines nested loops, filtering, string formatting, `enumerate`, and simple coordinate logic.


## Problem 18 — Generator expression comparison

A list comprehension immediately builds a list.
A generator expression produces values lazily.

Compute the same sum both ways and verify the results match.


In [18]:
limit = 100_000

sum_with_list = sum([n * n for n in range(limit) if n % 2 == 0])
sum_with_generator = sum(n * n for n in range(limit) if n % 2 == 0)

assert sum_with_list == sum_with_generator
sum_with_list


166661666700000

### Solution notes

When the only consumer is a function like `sum`, `any`, `all`, `min`, or `max`, a generator expression is often better because it avoids building an intermediate list.


## Final checklist

Use a list comprehension when:

- you are building a list
- the transformation is easy to understand
- the filter is easy to understand
- the comprehension fits comfortably on one or a few readable lines

Prefer a normal loop or helper functions when:

- you need several intermediate variables
- you need exception handling
- the comprehension is deeply nested
- readers would need to stop and mentally decode the expression
